## load_silver_fema_hazard
Conforms `bronze.fema_nri_counties` (county grain) into `silver.fact_fema_hazard_cbsa` (CBSA grain, geo-keyed, **no date** — a static hazard characteristic, joined to the housing facts on `geo_key`).

**Rollup:** trim + cast (Bronze keeps NRI leading-space padding); county cast failures → `silver.quarantine`. **INNER-join** the county→CBSA bridge — rural counties with no CBSA drop (CBSA-footprint coverage, *not* a quarantine case). Group by CBSA: `population`/`eal_valt` **SUM**; the intensive scores (`risk_score`, `sovi_score`, `resl_score`, the 10 per-hazard `*_risks`) **population-weighted mean** with a **per-measure denominator** (only counties non-null for that score contribute weight, so the ~88 NRI counties missing sovi/resl don't bias their means). Resolve `geo_key` via `dim_geo`; MERGE on `geo_key`. Scores-only — ratings (`*_riskr`/`sovi_ratng`/`resl_ratng`) are not carried. StepLog + transform_detail_log. Design: `weather_silver_gold_design.md` §3 (and §6.5 for the sovi/resl reconciliation).

In [ ]:
%run "../libs/notebook_init"

In [ ]:
# notebook_init injects: BRONZE, SILVER, CROSSWALKS, AUDIT, PIPELINE_RUN_ID, STATUS_*, StepLog,
# Utils, transform_detail_log_insert, spark, dbutils, F, datetime, timezone.

STEP_SEQUENCE = 1                                  # position owned by the orchestrator
SOURCE_SYSTEM = "fema_nri"
SOURCE_TABLE  = f"{BRONZE}.fema_nri_counties"
TARGET_TABLE  = f"{SILVER}.fact_fema_hazard_cbsa"
QUARANTINE    = f"{SILVER}.quarantine"
DIM_GEO       = f"{SILVER}.dim_geo"
COUNTY_BRIDGE = f"{CROSSWALKS}county_to_cbsa.csv"  # stcofips -> cbsa_code (committed reference)

# Intensive scores (0-100 percentile) -> POPULATION-WEIGHTED MEAN. risk_score is the composite;
# sovi/resl are the social-vulnerability / community-resilience scores; the 10 *_risks are the
# per-hazard scores. (eal_valt + population are extensive -> SUM, handled separately below.)
INTENSIVE = [
    "risk_score", "sovi_score", "resl_score",
    "hrcn_risks", "cfld_risks", "ifld_risks", "trnd_risks", "wfir_risks",
    "erqk_risks", "hail_risks", "swnd_risks", "hwav_risks", "wntw_risks",
]
# Every numeric source column we cast — the cast-error check covers all of them.
CAST_COLS = ["population", "eal_valt"] + INTENSIVE

In [ ]:
# Open the pipeline_step_log row (RUNNING). Closed explicitly by step.succeed() in the write
# cell, or by step.fail(e) in any work cell's 2-line handler.
nb = Utils.get_notebook_context(dbutils)
step = StepLog(
    spark, AUDIT, dbutils,
    pipeline_run_id = PIPELINE_RUN_ID,
    step_sequence   = STEP_SEQUENCE,
    notebook_folder = nb["notebook_folder"],
    notebook_name   = nb["notebook_name"],
    layer           = "silver",
    target_table    = TARGET_TABLE,
)
print(f"load_silver_fema_hazard: step_log_id={step.step_log_id}")

In [ ]:
# Read NRI counties (all-STRING), trim the NRI leading-space padding, cast every numeric column.
# cast_err (CLAUDE.md §11): a value that is non-null and non-blank but fails the numeric cast is a
# genuine failure; a blank just means "not reported" -> null, not an error. The double cast is a
# uniform numeric-validity test for all CAST_COLS. raw_payload keeps the full Bronze row for any
# quarantined county.
try:
    bronze = spark.table(SOURCE_TABLE)
    rows_read = bronze.count()

    cast_err = lambda col_name: (F.col(col_name).isNotNull()) & (F.trim(F.col(col_name)) != F.lit("")) \
                         & (F.trim(F.col(col_name)).cast("double").isNull())
    typed = bronze.select(
        F.col("stcofips"),
        F.trim(F.col("population")).cast("long").alias("population"),
        F.trim(F.col("eal_valt")).cast("double").alias("eal_valt"),
        *[F.trim(F.col(col_name)).cast("double").alias(col_name) for col_name in INTENSIVE],
        F.array_compact(F.array(*[F.when(cast_err(col_name), F.lit(col_name)) for col_name in CAST_COLS])).alias("cast_errors"),
        F.to_json(F.struct(*[F.col(col_name) for col_name in bronze.columns])).alias("raw_payload"),
    )
    step.rows_read = rows_read
    print(f"load_silver_fema_hazard: read {rows_read:,} NRI county rows")
except Exception as e:
    step.fail(e); raise

In [ ]:
# Quarantine county cast failures, roll the good counties up to CBSA, resolve geo_key, MERGE.
# §11.4 audit vars pre-declared. Quarantine is idempotent: DELETE this source's rows first, then
# append (cast failures at county grain; any unmatched CBSA post-rollup). Rural counties (no CBSA
# in the bridge) drop on the INNER join — that IS the CBSA-footprint coverage, not a quarantine
# case (design §3).
transform_started = datetime.now(timezone.utc)
rows_rejected = rows_inserted = rows_updated = 0
try:
    good = typed.where(F.size("cast_errors") == 0)
    bad  = typed.where(F.size("cast_errors") > 0)
    rows_rejected = bad.count()

    spark.sql(f"DELETE FROM {QUARANTINE} WHERE source_system = '{SOURCE_SYSTEM}'")
    if rows_rejected > 0:
        bad.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.col("stcofips").alias("natural_key"),
            F.col("raw_payload"),
            F.concat(F.lit("cast_failed:"), F.concat_ws(",", F.col("cast_errors"))).alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)

    # County -> CBSA bridge (committed reference). INNER join: rural counties drop (expected).
    bridge = spark.read.option("header", True).csv(COUNTY_BRIDGE)
    joined = good.join(bridge, "stcofips", "inner")

    # Population-weighted mean with a PER-MEASURE denominator: only counties whose score is
    # non-null contribute to that score's weight, so the ~88 counties missing sovi/resl don't
    # deflate those means. risk_score and the 10 *_risks are fully populated; same rule applies.
    def wmean(col_name):
        return (F.sum(F.col(col_name) * F.col("population"))
                / F.sum(F.when(F.col(col_name).isNotNull(), F.col("population")))).alias(col_name)

    rolled = joined.groupBy("cbsa_code").agg(
        F.sum("population").alias("population"),
        F.round(F.sum("eal_valt")).cast("long").alias("eal_valt"),
        *[wmean(col_name) for col_name in INTENSIVE],
    )

    # Resolve geo_key on cbsa_code. Null geo_key is unexpected (bridge + dim_geo are both OMB);
    # quarantine any such CBSA rather than drop it silently.
    geo = spark.table(DIM_GEO).select("geo_key", "cbsa_code")
    staged = rolled.join(geo, "cbsa_code", "left")
    matched   = staged.where(F.col("geo_key").isNotNull())
    unmatched = staged.where(F.col("geo_key").isNull())
    unmatched_n = unmatched.count()
    if unmatched_n > 0:
        unmatched.select(
            F.expr("uuid()").alias("quarantine_id"),
            F.lit(SOURCE_SYSTEM).alias("source_system"),
            F.lit(None).cast("string").alias("source_file_path"),
            F.col("cbsa_code").alias("natural_key"),
            F.to_json(F.struct(*rolled.columns)).alias("raw_payload"),
            F.lit("unmatched_geography").alias("quarantine_reason"),
            F.current_timestamp().alias("quarantined_ts"),
        ).write.format("delta").mode("append").saveAsTable(QUARANTINE)
        rows_rejected += unmatched_n

    fact_cols  = ["geo_key", "population", "eal_valt"] + INTENSIVE
    value_cols = ["population", "eal_valt"] + INTENSIVE
    matched.select(
        *[F.col(col_name) for col_name in fact_cols],
        F.current_timestamp().alias("inserted_ts"),
        F.current_timestamp().alias("updated_ts"),
    ).createOrReplaceTempView("fema_hazard_staging")

    set_clause = ", ".join(f"t.{col_name}=s.{col_name}" for col_name in value_cols) + ", t.updated_ts=s.updated_ts"
    cols_csv   = ", ".join(fact_cols + ["inserted_ts", "updated_ts"])
    vals_csv   = ", ".join(f"s.{col_name}" for col_name in fact_cols + ["inserted_ts", "updated_ts"])
    metrics = spark.sql(f"""
        MERGE INTO {TARGET_TABLE} t USING fema_hazard_staging s
        ON t.geo_key = s.geo_key
        WHEN MATCHED THEN UPDATE SET {set_clause}
        WHEN NOT MATCHED THEN INSERT ({cols_csv}) VALUES ({vals_csv})
    """).first().asDict()
    rows_inserted = metrics.get("num_inserted_rows") or 0
    rows_updated  = metrics.get("num_updated_rows") or 0

    step.rows_written = rows_inserted
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_SUCCEEDED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_written=rows_inserted, rows_inserted=rows_inserted, rows_updated=rows_updated,
        rows_rejected=rows_rejected, ended_timestamp=datetime.now(timezone.utc))
    step.succeed()
    print(f"load_silver_fema_hazard: inserted={rows_inserted:,} updated={rows_updated:,} "
          f"quarantined={rows_rejected:,} (read={step.rows_read:,})")
except Exception as e:
    transform_detail_log_insert(
        spark, AUDIT, PIPELINE_RUN_ID, step.step_log_id, SOURCE_TABLE, TARGET_TABLE,
        status=STATUS_FAILED, started_timestamp=transform_started, rows_read=step.rows_read,
        rows_rejected=rows_rejected, error_message=f"{type(e).__name__}: {e}",
        ended_timestamp=datetime.now(timezone.utc))
    step.fail(e); raise